# PPA Model Comparison - Phase 4 Week 8

Aggregates results from the 5 individual model notebooks (10 / 11 / 12 / 13 / 14)
plus the 2 baselines. Produces the final Friedman / Nemenyi / Wilcoxon tables and
the scorecard heatmap used for dual-champion selection.

**Assumes** each model notebook has saved its `metrics_df` to `outputs/metrics_<model>.csv`
with columns: `model, seed, fold, wmape, rmse_log, rmse_vol, mape, smape, train_time_sec`.
It also loads `outputs/elasticity_<model>.csv` (columns: sku, customer, beta_mean, beta_lo, beta_hi).

In [ ]:
import sys, os, glob
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.experiments import summarize
from src.stats_tests import (
    friedman_test, nemenyi_posthoc, wilcoxon_pairwise, critical_difference_diagram,
)
from src.elasticity import plausibility_scorecard
from src.config import OUTPUTS

## 1. Load per-model metrics (produced by notebooks 10-14 + baselines)

In [ ]:
metric_files = sorted(glob.glob(str(OUTPUTS / 'metrics_*.csv')))
print('found:', metric_files)
metrics_df = pd.concat([pd.read_csv(p) for p in metric_files], ignore_index=True)
print('shape:', metrics_df.shape)
metrics_df.head()

## 2. Summary table (mean +/- 95% CI across 20 obs per model)

In [ ]:
for metric in ['wmape', 'rmse_log', 'rmse', 'rmsle', 'mape', 'smape', 'r2']:
    print('\n==', metric, '==')
    print(summarize(metrics_df, metric))

## 3. Friedman omnibus test

In [ ]:
ft = friedman_test(metrics_df, metric='wmape')
print('chi2 =', ft['chi2'], 'p =', ft['p_value'])
print('average ranks (lower is better):')
for m, r in sorted(ft['avg_rank'].items(), key=lambda kv: kv[1]):
    print(f'  {m:20s} {r:.3f}')

## 4. Nemenyi post-hoc + Critical Difference diagram

In [ ]:
nem = nemenyi_posthoc(metrics_df, metric='wmape')
print(nem)
fig, ax = plt.subplots(figsize=(9, 2.5))
critical_difference_diagram(metrics_df, metric='wmape', ax=ax)
plt.show()

## 5. Pairwise Wilcoxon + Cohen's d

In [ ]:
wilcoxon_pairwise(metrics_df, metric='wmape')

## 6. Elasticity plausibility scorecard (the 35%-weight dimension)

In [ ]:
elast_files = sorted(glob.glob(str(OUTPUTS / 'elasticity_*.csv')))
rows = []
for p in elast_files:
    name = os.path.basename(p).replace('elasticity_', '').replace('.csv', '')
    e = pd.read_csv(p)
    s = plausibility_scorecard(e)
    s['model'] = name
    rows.append(s)
elast_scorecard = pd.DataFrame(rows).set_index('model')
elast_scorecard

## 7. Dual-champion recommendation

Combine the 5 scorecard dimensions (see plan) into a single recommendation.
The final write-up includes BOTH:
- **Academic claim** (with statistical significance asterisks on the WMAPE table)
- **Industrial deployment recommendation** (champion for elasticity dashboard + champion for simulator)